# 02 · Completion signaling on the hot path

CQ data vs flag-in-payload vs sentinel vs remote-write counter vs trailing READ, with per-post instruction counts.

In [ ]:
MODE = "local"            # "local": smoke runs on this machine (tcp / tcp;ofi_rxm / shm providers)
                          # "cluster": this host is the client, the server runs on PEER over ssh
PEER = "server1"         # ssh host for the server side (cluster mode)
PEER_IP = "192.168.1.242" # server address on the RDMA netdev; default taken from benchmark_dual_qp_remote.py, check it
REMOTE_REPO = None        # repo path on PEER if it differs from this host
DOMAIN = "rxe0"           # RDMA device / libfabric domain on both nodes
SV_GID_INDEX = None       # set if the auto-picked GID (first IPv4-mapped) is wrong, e.g. "1"
HOT_CPU = None            # pin the hot loop (cluster: pick a core away from the NIC IRQs)
COLD_CPU = None           # pin the cold stream thread
ITERS = None              # None: 2000 local, 20000 cluster
RESULTS_SUBDIR = "fabric"


In [ ]:
import os, sys, json, socket, subprocess, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "middleware").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
HOST = socket.gethostname()
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS = (ROOT / "benchmarks" / "results" / RESULTS_SUBDIR) if MODE == "cluster" else Path("/tmp/scuffed-local") / RESULTS_SUBDIR
RESULTS.mkdir(parents=True, exist_ok=True)
def save(name, obj):
    p = RESULTS / f"{HOST}_{name}.json"
    obj = {"host": HOST, "utc": STAMP, "mode": MODE, **obj}
    p.write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", p)
    return p
def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=ROOT)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode:
        raise RuntimeError(f"{cmd} -> {r.returncode}")
    return r
print("repo:", ROOT, "| host:", HOST, "| mode:", MODE, "| results ->", RESULTS)

Update 7, next step 3: how should the receiver learn that a hot write landed?

| mechanism | initiator | target detects by | cost / caveat |
|---|---|---|---|
| `cq` | `fi_writedata` / WRITE_WITH_IMM | polling the RX CQ for remote CQ data | consumes a posted receive per message |
| `flag` | plain write, last 8 B = sequence | spinning on memory | relies on in-order placement of the last bytes (holds per packet on RC; not a spec guarantee for multi-packet messages) |
| `sentinel` | payload write, then 8 B sentinel write | spinning on the sentinel | needs write-after-write ordering on the endpoint |
| `cntr` | plain write | `FI_REMOTE_WRITE` counter | libfabric only; providers that do not count remote writes are detected by a probe and skipped |
| `read` | write then 8 B read | (initiator) read completion | one-way delivery confirmation, the GPFS flush pattern; not a ping-pong |

`--perf` adds user-space instructions retired per post (perf_event_open), the per-operation overhead metric
from Papadopoulou et al. Adopt a mechanism only if it moves p99.

In [ ]:
from benchmarks.fabric import runner
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
CLUSTER = MODE == "cluster"
N = ITERS or (20000 if CLUSTER else 2000)
W = max(100, N // 20)
PAIR = dict(peer=PEER, peer_ip=PEER_IP, remote_repo=REMOTE_REPO) if CLUSTER else {}
ENV = {"SV_GID_INDEX": str(SV_GID_INDEX)} if SV_GID_INDEX is not None else {}
STACKS = runner.stacks(MODE)
if CLUSTER:
    for st in STACKS:
        if st["tool"] == "sf_bench":
            st["args"] += ["-d", DOMAIN]
        else:
            ENV.setdefault("SV_IB_PORT", "1")
PIN = (["--hot-cpu", str(HOT_CPU)] if HOT_CPU is not None else []) + (["--cold-cpu", str(COLD_CPU)] if COLD_CPU is not None else [])
print(runner.build(remote=PEER if CLUSTER else None, remote_repo=REMOTE_REPO))
for st in STACKS: print(st["label"], st["tool"], st["args"])

In [ ]:
MECHS = ["cq", "flag", "sentinel", "cntr", "read"]
SIZES = "8,64,512,4096,16384" if CLUSTER else "64,4096"
raw, rows = [], []
for st in STACKS:
    for m in MECHS:
        d = runner.try_pair(st["tool"], st["args"] + ["-m", "sig", "-M", m, "-n", str(N), "-w", str(W), "-S", SIZES, "--perf"], env=ENV, **PAIR)
        raw.append(d); rows += runner.rows_sig(d, st["label"])
df = pd.DataFrame(rows)
df[[c for c in ["label", "mech", "size", "p50_us", "p99_us", "p999_us", "post_instructions_mean", "metric", "skipped", "error"] if c in df]]

In [ ]:
save("completion_signaling", {"rows": rows, "raw": raw})
ok = df.dropna(subset=["p50_us"]) if "p50_us" in df else df
labels = list(dict.fromkeys(ok.label))
fig, axes = plt.subplots(1, len(labels), figsize=(6 * len(labels), 4.5), squeeze=False)
for ax, lab in zip(axes[0], labels):
    for m, g in ok[ok.label == lab].groupby("mech"):
        ax.plot(g["size"], g["p99_us"], marker="o", label=f"{m} p99")
        ax.plot(g["size"], g["p50_us"], ls="--", alpha=.5)
    ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_title(lab, fontsize=9)
    ax.set_xlabel("payload (B)"); ax.set_ylabel("us (solid p99, dashed p50)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout()